# A3-M2 · HNSW index factory
교안의 같은 번호 단원과 함께 봅니다.

## 08-1. 작은 그래프 만들기 — 점 30개, M=4

In [1]:
import faiss, heapq
import numpy as np

faiss.omp_set_num_threads(1)                                  # 실행할 때마다 같은 그래프가 나오도록 스레드 1개
rng = np.random.default_rng(7)
pts = (rng.random((30, 2)) * 10).round(1).astype("float32")   # 0~10 사이 2차원 점 30개
hnsw = faiss.IndexHNSWFlat(2, 4)                              # M=4: 위층에서 이웃 최대 4개(맨 아래층은 2M=8개)
hnsw.hnsw.efConstruction = 20
hnsw.add(pts)

h = hnsw.hnsw
levels = faiss.vector_to_array(h.levels) - 1                 # 점마다 올라가 있는 최고 층
print("최고 층:", h.max_level, "| 입구(entry point):", h.entry_point)
for L in range(h.max_level, -1, -1):
    on = np.where(levels >= L)[0]
    print(f"층 {L}: {len(on)}개", on.tolist() if L > 0 else "(모든 점)")

최고 층: 2 | 입구(entry point): 12
층 2: 3개 [11, 12, 22]
층 1: 9개 [0, 1, 7, 11, 12, 15, 21, 22, 29]
층 0: 30개 (모든 점)


## 08-2. 이웃 목록 들여다보기

In [2]:
offsets = faiss.vector_to_array(h.offsets)
nbrs = faiss.vector_to_array(h.neighbors)
cum = faiss.vector_to_array(h.cum_nneighbor_per_level)

def neighbors(i, level):                               # i번 점의 level층 이웃
    b, e = int(offsets[i] + cum[level]), int(offsets[i] + cum[level + 1])
    return [int(v) for v in nbrs[b:e] if v >= 0]       # 빈칸은 -1

for i in [h.entry_point, 22, 7]:
    print(f"{i}번 {pts[i].astype(float).round(1).tolist()} (최고 {levels[i]}층)")
    for L in range(levels[i], -1, -1):
        print(f"   층 {L} 이웃: {neighbors(i, L)}")

12번 [0.4, 5.1] (최고 2층)
   층 2 이웃: [11, 22]
   층 1 이웃: [11, 22, 21, 7]
   층 0 이웃: [26, 20, 23]
22번 [6.4, 7.4] (최고 2층)
   층 2 이웃: [11, 12]
   층 1 이웃: [0, 7, 29]
   층 0 이웃: [0, 14, 8]
7번 [5.0, 5.5] (최고 1층)
   층 1 이웃: [22, 15, 12]
   층 0 이웃: [22, 15, 4, 12, 6, 25, 14]


## 08-3. HNSW 검색을 손으로 따라가기

In [3]:
q = np.array([5.0, 5.0], dtype="float32")
dist = lambda i: float(((pts[i] - q) ** 2).sum())
ndis = 1
cur = int(h.entry_point)
print(f"출발: 입구 {cur}번 거리 {dist(cur):.2f}")
for L in range(h.max_level, 0, -1):                   # 위층: 더 가까운 이웃이 없을 때까지 한 칸씩
    while True:
        best = cur
        for nb in neighbors(cur, L):
            ndis += 1
            if dist(nb) < dist(best):
                best = nb
        print(f"  층 {L}: {cur}번({dist(cur):.2f}) → {best}번({dist(best):.2f})" + ("  멈춤, 아래층으로" if best == cur else ""))
        if best == cur:
            break
        cur = best

def search_layer0(ep, ef):                            # 맨 아래층: 후보를 ef개까지 넓게 들고 탐색
    global ndis
    visited = {ep}
    cand = [(dist(ep), ep)]                            # 앞으로 볼 후보 (가까운 순)
    top = [(-dist(ep), ep)]                            # 지금까지 찾은 좋은 점 ef개 (먼 것이 맨 위)
    while cand:
        dc, c = heapq.heappop(cand)
        if len(top) >= ef and dc > -top[0][0]:
            break                                      # 남은 후보가 다 지금 결과보다 멀면 끝
        for nb in neighbors(c, 0):
            if nb in visited:
                continue
            visited.add(nb); ndis += 1
            dn = dist(nb)
            if len(top) < ef or dn < -top[0][0]:
                heapq.heappush(cand, (dn, nb)); heapq.heappush(top, (-dn, nb))
                if len(top) > ef:
                    heapq.heappop(top)
    return sorted((-dneg, i) for dneg, i in top), visited

res, visited = search_layer0(cur, ef=8)
print("  층 0: 방문한 점", len(visited), "개 →", sorted(visited))
print("손으로 구한 top-3:", [(i, round(dd, 2)) for dd, i in res[:3]], "| 거리 계산", ndis, "번")

hnsw.hnsw.efSearch = 8
faiss.cvar.hnsw_stats.reset()
D, I = hnsw.search(q[None], 3)
print("faiss 결과      :", I[0].tolist(), D[0].round(2).tolist(), "| 거리 계산", faiss.cvar.hnsw_stats.ndis, "번")
flat = faiss.IndexFlatL2(2); flat.add(pts)
print("전수 비교 정답  :", flat.search(q[None], 3)[1][0].tolist(), "| 거리 계산 30번")

출발: 입구 12번 거리 21.17
  층 2: 12번(21.17) → 22번(7.72)
  층 2: 22번(7.72) → 22번(7.72)  멈춤, 아래층으로
  층 1: 22번(7.72) → 7번(0.25)
  층 1: 7번(0.25) → 7번(0.25)  멈춤, 아래층으로
  층 0: 방문한 점 23 개 → [0, 1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 14, 15, 17, 21, 22, 23, 25, 26, 27, 28, 29]
손으로 구한 top-3: [(7, 0.25), (14, 1.7), (25, 2.96)] | 거리 계산 33 번
faiss 결과      : [7, 14, 25] [0.25, 1.7000000476837158, 2.9600000381469727] | 거리 계산 32 번
전수 비교 정답  : [7, 14, 25] | 거리 계산 30번


## 08-4. efSearch를 바꾸면 — 작은 그래프에서

In [4]:
for ef in [1, 3, 8, 30]:
    hnsw.hnsw.efSearch = ef
    faiss.cvar.hnsw_stats.reset()
    D, I = hnsw.search(q[None], 3)
    print(f"efSearch={ef:<2} → {I[0].tolist()}  거리 계산 {faiss.cvar.hnsw_stats.ndis}번")

efSearch=1  → [7, 14, 25]  거리 계산 17번
efSearch=3  → [7, 14, 25]  거리 계산 20번
efSearch=8  → [7, 14, 25]  거리 계산 32번
efSearch=30 → [7, 14, 25]  거리 계산 39번


## 08-5. 2만 개 실험 — efSearch별 정확도·거리 계산 수

In [5]:
import faiss, time
import numpy as np

rng = np.random.default_rng(0)
d, n, k = 64, 20000, 10
centers = rng.normal(0, 10, (50, d)).astype("float32")          # 숨은 주제 50개
xb = (centers[rng.integers(0, 50, n)] + rng.normal(0, 1, (n, d))).astype("float32")   # 문서 2만 개
xq = (centers[rng.integers(0, 50, 100)] + rng.normal(0, 1, (100, d))).astype("float32")  # 질문 100개

flat = faiss.IndexFlatL2(d)
flat.add(xb)
t = time.perf_counter(); _, I_true = flat.search(xq, k); t_flat = time.perf_counter() - t

def recall(I):                     # 정답(전수 비교) k개 중 몇 개를 찾았나, 평균
    return np.mean([len(set(I[i]) & set(I_true[i])) / k for i in range(len(I))])

print(f"문서 {xb.shape}, 질문 {xq.shape}, Flat 검색 {t_flat*1000:.1f} ms (질문 100개)")

import os
faiss.omp_set_num_threads(os.cpu_count())                     # 큰 데이터는 다시 여러 스레드로
hnsw20 = faiss.IndexHNSWFlat(d, 32)                   # M=32
print("efConstruction:", hnsw20.hnsw.efConstruction, "| efSearch:", hnsw20.hnsw.efSearch)
t = time.perf_counter(); hnsw20.add(xb); print(f"만들기 {time.perf_counter()-t:.2f}초 (학습 단계 없음)")
for ef in [16, 32, 64, 128]:
    hnsw20.hnsw.efSearch = ef
    faiss.cvar.hnsw_stats.reset()
    t = time.perf_counter(); _, I = hnsw20.search(xq, k); el = time.perf_counter() - t
    print(f"efSearch={ef:<3} recall={recall(I):.3f}  질문당 거리계산 {faiss.cvar.hnsw_stats.ndis/len(xq):5.0f}번 (Flat {n})  {el*1000:5.1f} ms")

문서 (20000, 64), 질문 (100, 64), Flat 검색 138.3 ms (질문 100개)
efConstruction: 40 | efSearch: 16
만들기 1.94초 (학습 단계 없음)
efSearch=16  recall=0.981  질문당 거리계산   374번 (Flat 20000)   32.4 ms
efSearch=32  recall=0.989  질문당 거리계산   433번 (Flat 20000)    3.8 ms
efSearch=64  recall=0.989  질문당 거리계산   466번 (Flat 20000)   19.4 ms
efSearch=128 recall=0.999  질문당 거리계산   481번 (Flat 20000)    9.9 ms


## 08-6. M과 efConstruction — 만들 때 정하는 값

In [6]:
for M in [8, 16, 32]:
    for efc in [16, 64]:
        ix = faiss.IndexHNSWFlat(d, M)
        ix.hnsw.efConstruction = efc                   # add 전에 정해야 한다
        t = time.perf_counter(); ix.add(xb); t_build = time.perf_counter() - t
        ix.hnsw.efSearch = 32
        _, I = ix.search(xq, k)
        mb = faiss.serialize_index(ix).nbytes / 1e6
        print(f"M={M:<2} efConstruction={efc:<3} 만들기 {t_build:4.2f}초  크기 {mb:5.2f} MB  recall(efSearch=32) {recall(I):.3f}")

M=8  efConstruction=16  만들기 1.02초  크기  6.73 MB  recall(efSearch=32) 0.867
M=8  efConstruction=64  만들기 1.55초  크기  6.73 MB  recall(efSearch=32) 0.885
M=16 efConstruction=16  만들기 0.54초  크기  8.00 MB  recall(efSearch=32) 0.908
M=16 efConstruction=64  만들기 1.08초  크기  8.00 MB  recall(efSearch=32) 0.986
M=32 efConstruction=16  만들기 0.70초  크기 10.56 MB  recall(efSearch=32) 0.952
M=32 efConstruction=64  만들기 1.14초  크기 10.56 MB  recall(efSearch=32) 0.993


## 09-1. index_factory — 문자열로 조립하고 크기 비교

In [7]:
for spec in ["Flat", "IDMap,Flat", "IVF100,Flat", "IVF100,PQ16", "IVF100,SQ8", "HNSW32"]:
    ix = faiss.index_factory(d, spec)
    print(f"{spec:<12} → {type(ix).__name__:<26} 학습 필요: {not ix.is_trained}")

ix = faiss.index_factory(d, "IVF100,Flat", faiss.METRIC_INNER_PRODUCT)
print("내적 IVF metric_type:", ix.metric_type, "(0=내적, 1=L2)")

sizes = {}
for spec in ["Flat", "IVF100,Flat", "IVF100,SQ8", "IVF100,PQ32", "HNSW32"]:
    ix = faiss.index_factory(d, spec)
    if not ix.is_trained:
        ix.train(xb)
    ix.add(xb)
    sizes[spec] = faiss.serialize_index(ix).nbytes / 1e6
    print(f"{spec:<12} {sizes[spec]:6.2f} MB")

Flat         → IndexFlat                  학습 필요: False
IDMap,Flat   → IndexIDMap                 학습 필요: False
IVF100,Flat  → IndexIVFFlat               학습 필요: True
IVF100,PQ16  → IndexIVFPQ                 학습 필요: True
IVF100,SQ8   → IndexIVFScalarQuantizer    학습 필요: True
HNSW32       → IndexHNSWFlat              학습 필요: False
내적 IVF metric_type: 0 (0=내적, 1=L2)
Flat           5.12 MB
IVF100,Flat    5.31 MB
IVF100,SQ8     1.47 MB
IVF100,PQ32    0.89 MB
HNSW32        10.56 MB
